# Lesson 3.4 — Renaming, Adding, and Dropping Columns

**Objectives**
- Rename columns with `.rename()` and by reassigning `.columns`
- Create new columns from existing ones (vectorized operations, `.apply()`)
- Drop columns/rows with `.drop()`

See `modules/03-pandas/notes.md` (Lesson 3.4) for the full written explanation.


In [1]:
import numpy as np
import pandas as pd
from data_science_course.datasets import load_customers, load_orders

customers = load_customers()
orders = load_orders()

## Renaming columns

In [2]:
renamed = customers.rename(columns={"acquisition_channel": "channel"})
renamed.columns

Index(['customer_id', 'signup_date', 'region', 'age', 'membership_tier',
       'channel', 'churned'],
      dtype='object')

In [3]:
upper_cols = renamed.copy()
upper_cols.columns = [c.upper() for c in upper_cols.columns]
upper_cols.columns

Index(['CUSTOMER_ID', 'SIGNUP_DATE', 'REGION', 'AGE', 'MEMBERSHIP_TIER',
       'CHANNEL', 'CHURNED'],
      dtype='object')

## Cleaning up `region` for real

In [4]:
customers["region"].value_counts(dropna=False)

region
East      101
east       86
South      78
south      75
west       72
WEST       70
West       58
north      53
SOUTH      51
 North     51
NORTH      41
North      40
NaN        24
Name: count, dtype: int64

In [5]:
customers["region"] = (
    customers["region"]
    .str.strip()
    .str.lower()
    .str.title()
)
customers["region"].value_counts(dropna=False)

region
South    204
West     200
East     187
North    185
NaN       24
Name: count, dtype: int64

Down to exactly the four canonical regions plus genuinely-missing rows. This is a
**vectorized** string operation -- it runs on all 800 rows at once, no Python `for`
loop needed.

## Adding new columns

In [6]:
orders["revenue_per_unit"] = orders["order_total"] / orders["quantity"]
orders[["order_id", "quantity", "order_total", "revenue_per_unit"]].head(3)

,order_id,quantity,order_total,revenue_per_unit
0,O000080,1,72.48,72.48
1,O002751,1,69.18,69.18
2,O003297,1,201.03,201.03


In [7]:
def size_bucket(total: float) -> str:
    if total < 50:
        return "small"
    elif total < 150:
        return "medium"
    return "large"

orders["order_size"] = orders["order_total"].apply(size_bucket)
orders["order_size"].value_counts()

order_size
large     2631
small     1742
medium    1636
Name: count, dtype: int64

`.apply()` calls `size_bucket` once per row -- flexible, but slower than a vectorized
expression because it falls back to a Python function call per value. Prefer vectorized
math/`.str`/comparisons whenever the logic allows it; save `.apply()` for genuinely
custom per-row logic like this multi-branch bucketing.

In [8]:
orders["is_discounted"] = np.where(orders["discount_pct"] > 0, "yes", "no")
orders["is_discounted"].value_counts(dropna=False)

is_discounted
no     4086
yes    1923
Name: count, dtype: int64

`np.where` is a fast, vectorized if/else -- a good alternative to `.apply()` for
simple two-branch logic. Note it counted `NaN` `discount_pct` rows as `"no"` (since
`NaN > 0` is `False`) -- worth remembering once we discuss missing `discount_pct`
properly in Lesson 3.5.

## Dropping columns and rows

In [9]:
trimmed = orders.drop(columns=["revenue_per_unit", "order_size", "is_discounted"])
trimmed.columns

Index(['order_id', 'customer_id', 'product_id', 'order_date', 'quantity',
       'unit_price', 'discount_pct', 'payment_method', 'shipping_region',
       'status', 'order_total'],
      dtype='object')

In [10]:
orders_no_row0 = orders.drop(index=0)
len(orders_no_row0) == len(orders) - 1

True

## Removing the 5 duplicate `order_id` rows

In [11]:
print(orders.shape)
orders = orders.drop_duplicates(subset=["order_id"], keep="first")
print(orders.shape)

(6009, 14)
(6004, 14)


5 rows gone, matching `data/README.md` exactly. `keep="first"` keeps the first
occurrence of each `order_id` and drops the rest.

## Try it yourself

1. Rename `orders`' `order_total` column to `total_amount` (without mutating the
   original `orders`).
2. Add a column `discount_amount` to `orders`, computed as
   `unit_price * quantity * discount_pct` (treat missing `discount_pct` as producing
   `NaN` -- don't fill it yet, that's Lesson 3.5).
3. Add a column `is_bulk_order` that's `True` when `quantity >= 3`, using a vectorized
   comparison (no `.apply()` needed).
4. Drop the `payment_method` and `shipping_region` columns from a **copy** of `orders`
   and confirm the original `orders` still has them.


In [12]:
# 1. TODO


# 2. TODO


# 3. TODO


# 4. TODO


### Solution

In [13]:
# 1.
orders_renamed = orders.rename(columns={"order_total": "total_amount"})
print("total_amount" in orders_renamed.columns, "order_total" in orders.columns)

# 2.
orders["discount_amount"] = orders["unit_price"] * orders["quantity"] * orders["discount_pct"]
print(orders[["order_id", "discount_amount"]].head(3))

# 3.
orders["is_bulk_order"] = orders["quantity"] >= 3
print(orders["is_bulk_order"].value_counts())

# 4.
trimmed_copy = orders.drop(columns=["payment_method", "shipping_region"])
print("payment_method" in trimmed_copy.columns)  # False
print("payment_method" in orders.columns)        # True -- original untouched

True True
  order_id  discount_amount
0  O000080           3.8145
1  O002751          17.2940
2  O003297          67.0100
is_bulk_order
False    5411
True      593
Name: count, dtype: int64
False
True
